In [1]:
pip install pulp

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

food_candidates = pd.read_csv("./ready_food_data_frame.csv")


In [3]:
patient_target = {
    "diet_type": "DM_HT_OBESITY",
    "energy_kcal": 1600,
    "carb_g": 240,
    "protein_g": 60,
    "fat_g": 44,
    "sodium_mg_max": 2000,
    "fiber_g_min": 25
}

In [4]:
def filter_foods_for_disease(df, diet_type):
    filtered = df.copy()

    # DM: remove sugar category if exists
    if "DM" in diet_type:
        filtered = filtered[filtered["category_code"] != "G"]

    # Hypertension: remove very high sodium foods
    if "HT" in diet_type:
        filtered = filtered[
            filtered["sodium_mg_per_portion"].isna() |
            (filtered["sodium_mg_per_portion"] <= 400)
        ]

    # Obesity: remove very high energy per portion
    if "OBESITY" in diet_type:
        filtered = filtered[
            filtered["energy_kcal_per_portion"] <= 250
        ]

    return filtered


candidate_for_patient = filter_foods_for_disease(
    food_candidates,
    patient_target["diet_type"]
)

print("Candidate foods after disease filter:", len(candidate_for_patient))

display(
    candidate_for_patient[[
        "food_code",
        "food_name",
        "category_code",
        "urt",
        "gram_per_portion",
        "energy_kcal_per_portion",
        "carb_g_per_portion",
        "sodium_mg_per_portion",
        "fiber_g_per_portion"
    ]].head(20)
)

Candidate foods after disease filter: 93


,food_code,food_name,category_code,urt,gram_per_portion,energy_kcal_per_portion,carb_g_per_portion,sodium_mg_per_portion,fiber_g_per_portion
5,AP006,"Bihun, mentah",MP,1/2 Gelas,50.0,174.00,41.050,2.50,0.600
6,AP020,"Makaroni, mentah",MP,1½ Gelas,50.0,176.50,39.350,2.50,2.450
7,AP024,Roti putih,MP,3 Iris,70.0,173.60,35.000,63.70,6.370
8,AP029,Biskuit,MP,4 Buah Besar,40.0,183.20,30.040,8.12,0.840
9,BR013,"Kentang, segar",MP,2 Buah Sedang,210.0,130.20,28.350,14.70,1.050
10,BR014,"Kentang hitam, segar",MP,12 Biji,125.0,177.50,42.125,27.50,6.750
11,CR018,"Kacang kedelai, segar",LN,2 Sendok Makan,25.0,71.50,7.525,7.00,0.725
12,CR027,"Kacang merah, segar",LN,2 Sendok Makan,25.0,42.75,7.000,1.75,0.525
13,CP005,"kacang hijau, rebus",LN,2 Sendok Makan,25.0,27.25,4.575,111.75,0.375
14,CP007,"Kacang kedelai, rebus",LN,2 Sendok Makan,25.0,47.25,3.175,45.25,0.400


In [5]:
import pandas as pd
import numpy as np
import pulp

# ============================================================
# 1. Prepare MILP input data
# ============================================================

milp_df = candidate_for_patient.copy().reset_index(drop=True)

# Required nutrient columns
required_cols = [
    "food_name",
    "category_code",
    "urt",
    "gram_per_portion",
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "fiber_g_per_portion"
]

# Keep only rows with required values
milp_df = milp_df.dropna(subset=[
    "category_code",
    "energy_kcal_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "fiber_g_per_portion"
]).copy()

# Make sure numeric columns are numeric
numeric_cols = [
    "gram_per_portion",
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

for col in numeric_cols:
    if col in milp_df.columns:
        milp_df[col] = pd.to_numeric(milp_df[col], errors="coerce").fillna(0)

print("Total MILP candidate foods:", len(milp_df))

display(
    milp_df.groupby("category_code")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

Total MILP candidate foods: 86


,category_code,count
5,S,40
0,B,17
1,LH,9
2,LN,7
4,MP,6
3,M,5
6,SS,2


In [6]:
# ============================================================
# 2. Create MILP model
# ============================================================

model = pulp.LpProblem("DM_HT_Obesity_Menu_Recommendation", pulp.LpMinimize)

# Allow half-portion:
# portion_step = 0.5 means:
# x[i] = 0 -> 0 portion
# x[i] = 1 -> 0.5 portion
# x[i] = 2 -> 1 portion
# x[i] = 3 -> 1.5 portions
# x[i] = 4 -> 2 portions

portion_step = 0.5
max_portion_per_food = 2.0

x = {
    i: pulp.LpVariable(
        f"x_{i}",
        lowBound=0,
        upBound=int(max_portion_per_food / portion_step),
        cat="Integer"
    )
    for i in milp_df.index
}

# Real portion used in calculation
portion = {
    i: x[i] * portion_step
    for i in milp_df.index
}


Formulation total of nutrion in daily menu based on combination from MILP

In [7]:
# ============================================================
# 3. Define nutrient totals
# ============================================================

total_energy = pulp.lpSum(
    portion[i] * milp_df.loc[i, "energy_kcal_per_portion"]
    for i in milp_df.index
)

total_protein = pulp.lpSum(
    portion[i] * milp_df.loc[i, "protein_g_per_portion"]
    for i in milp_df.index
)

total_fat = pulp.lpSum(
    portion[i] * milp_df.loc[i, "fat_g_per_portion"]
    for i in milp_df.index
)

total_carb = pulp.lpSum(
    portion[i] * milp_df.loc[i, "carb_g_per_portion"]
    for i in milp_df.index
)

total_sodium = pulp.lpSum(
    portion[i] * milp_df.loc[i, "sodium_mg_per_portion"]
    for i in milp_df.index
)

total_fiber = pulp.lpSum(
    portion[i] * milp_df.loc[i, "fiber_g_per_portion"]
    for i in milp_df.index
)

total_potassium = pulp.lpSum(
    portion[i] * milp_df.loc[i, "potassium_mg_per_portion"]
    for i in milp_df.index
) if "potassium_mg_per_portion" in milp_df.columns else 0

total_calcium = pulp.lpSum(
    portion[i] * milp_df.loc[i, "calcium_mg_per_portion"]
    for i in milp_df.index
) if "calcium_mg_per_portion" in milp_df.columns else 0

In [8]:
portion_plan = {
    1100: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 1, "SS": 0, "M": 3, "G": 1},
    1200: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 1, "SS": 1, "M": 3, "G": 2},
    1300: {"MP": 3, "LH": 2, "LN": 2, "S": 2, "B": 2, "SS": 1, "M": 4, "G": 2},
    1400: {"MP": 3, "LH": 3, "LN": 3, "S": 2, "B": 2, "SS": 0, "M": 3, "G": 3},
    1500: {"MP": 3, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 0, "M": 4, "G": 3},
    1600: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 2, "SS": 0, "M": 4, "G": 2},
    1700: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 3, "G": 2},
    1800: {"MP": 4, "LH": 3, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 4, "G": 3},
    1900: {"MP": 4, "LH": 4, "LN": 3, "S": 3, "B": 3, "SS": 1, "M": 4, "G": 3},
    2000: {"MP": 4, "LH": 3, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 4, "G": 4},
    2100: {"MP": 4, "LH": 3, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 4, "G": 4},
    2200: {"MP": 4, "LH": 3, "LN": 3, "S": 4, "B": 4, "SS": 2, "M": 5, "G": 4},
    2300: {"MP": 5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 0, "M": 5, "G": 4},
    2400: {"MP": 5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 5, "G": 4},
    2500: {"MP": 5.5, "LH": 4, "LN": 4, "S": 4, "B": 4, "SS": 1, "M": 5, "G": 4},
}

In [9]:
def get_disease_constraints(diet_type, energy_kcal):
    """
    diet_type options:
    HT
    HT_OBESITY
    DM
    DM_OBESITY
    DM_HT
    DM_HT_OBESITY
    """

    constraints = {
        "energy_target": energy_kcal,
        "energy_mode": "maintenance",
        "carb_min_g": None,
        "carb_max_g": None,
        "fat_min_g": None,
        "fat_max_g": None,
        "sodium_max_mg": None,
        "potassium_min_mg": None,
        "calcium_min_mg": None,
        "fiber_min_g": None,
        "vegetable_fruit_min_portions": None,
        "sugar_max_portions": None
    }

    # 1. Hipertensi tanpa obesitas
    if diet_type == "HT":
        constraints["potassium_min_mg"] = 4700
        constraints["calcium_min_mg"] = 800

    # 2. Hipertensi dengan obesitas
    elif diet_type == "HT_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["carb_min_g"] = energy_kcal * 0.55 / 4
        constraints["carb_max_g"] = energy_kcal * 0.65 / 4
        constraints["fat_min_g"] = energy_kcal * 0.20 / 9
        constraints["fat_max_g"] = energy_kcal * 0.25 / 9
        constraints["sodium_max_mg"] = 2300
        constraints["potassium_min_mg"] = 4700
        constraints["calcium_min_mg"] = 800

    # 3. DM tanpa obesitas
    elif diet_type == "DM":
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sugar_max_portions"] = 0

    # 4. DM dengan obesitas
    elif diet_type == "DM_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    # 5. DM + Hipertensi tanpa obesitas
    elif diet_type == "DM_HT":
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sodium_max_mg"] = 2300
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    # 6. DM + Hipertensi + Obesitas
    elif diet_type == "DM_HT_OBESITY":
        constraints["energy_mode"] = "deficit"
        constraints["carb_min_g"] = 130
        constraints["fiber_min_g"] = 25 * (energy_kcal / 1000)
        constraints["sodium_max_mg"] = 2000
        constraints["vegetable_fruit_min_portions"] = 5
        constraints["sugar_max_portions"] = 0

    return constraints

In [10]:
# ============================================================
# 3. Pick nearest available energy level
# ============================================================

def get_nearest_energy_level(energy_kcal, available_levels):
    return min(available_levels, key=lambda x: abs(x - energy_kcal))


def adjust_portion_plan_for_disease(base_plan, diet_type):
    plan = base_plan.copy()

    # DM: sugar should be avoided or minimized
    if "DM" in diet_type:
        plan["G"] = 0

    # Obesity: reduce sugar and oil portions
    if "OBESITY" in diet_type:
        plan["G"] = 0
        if "M" in plan:
            plan["M"] = max(0, plan["M"] - 1)

    # DM obesity / DM+HT: ensure vegetable + fruit >= 5 portions
    if diet_type in ["DM_OBESITY", "DM_HT", "DM_HT_OBESITY"]:
        current_sb = plan.get("S", 0) + plan.get("B", 0)
        if current_sb < 5:
            plan["S"] = plan.get("S", 0) + (5 - current_sb)

    return plan

In [11]:
# ============================================================
# 5. Example patient target
# ============================================================

diet_type = "DM_HT_OBESITY"
energy_target = 1600

constraints = get_disease_constraints(diet_type, energy_target)

nearest_energy = get_nearest_energy_level(
    energy_target,
    list(portion_plan.keys())
)

base_plan = portion_plan[nearest_energy]
adjusted_plan = adjust_portion_plan_for_disease(base_plan, diet_type)

candidate_for_patient = filter_foods_for_disease(food_candidates, diet_type)

print("Diet type:", diet_type)
print("Energy target:", energy_target)
print("Nearest portion plan:", nearest_energy)
print("\nDisease constraints:")
print(constraints)

print("\nBase portion plan:")
print(base_plan)

print("\nAdjusted portion plan:")
print(adjusted_plan)

print("\nTotal food candidates before filter:", len(food_candidates))
print("Total food candidates after disease filter:", len(candidate_for_patient))

category_after_filter = (
    candidate_for_patient
    .groupby("category_code")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(category_after_filter)

Diet type: DM_HT_OBESITY
Energy target: 1600
Nearest portion plan: 1600

Disease constraints:
{'energy_target': 1600, 'energy_mode': 'deficit', 'carb_min_g': 130, 'carb_max_g': None, 'fat_min_g': None, 'fat_max_g': None, 'sodium_max_mg': 2000, 'potassium_min_mg': None, 'calcium_min_mg': None, 'fiber_min_g': 40.0, 'vegetable_fruit_min_portions': 5, 'sugar_max_portions': 0}

Base portion plan:
{'MP': 4, 'LH': 3, 'LN': 3, 'S': 3, 'B': 2, 'SS': 0, 'M': 4, 'G': 2}

Adjusted portion plan:
{'MP': 4, 'LH': 3, 'LN': 3, 'S': 3, 'B': 2, 'SS': 0, 'M': 3, 'G': 0}

Total food candidates before filter: 99
Total food candidates after disease filter: 93


,category_code,count
5,S,40
0,B,19
1,LH,12
2,LN,8
4,MP,6
3,M,5
6,SS,3


In [12]:
# ============================================================
# 4. Category portion constraints
# ============================================================

for category, required_portion in adjusted_plan.items():

    category_indices = milp_df[
        milp_df["category_code"] == category
    ].index.tolist()

    # If category has no food and required portion is 0, skip safely
    if len(category_indices) == 0 and required_portion == 0:
        continue

    # If category has no food but required portion > 0, this is a real problem
    if len(category_indices) == 0 and required_portion > 0:
        print(
            f"Warning: no candidate food for category {category}, "
            f"but required portion is {required_portion}"
        )
        continue

    model += (
        pulp.lpSum(portion[i] for i in category_indices) == required_portion,
        f"portion_{category}"
    )

Ensure Constrain nutrion maxiumum around 10 percent

In [13]:
# ============================================================
# 5. Disease-specific nutrient constraints
# ============================================================

energy_target = constraints["energy_target"]
energy_min = energy_target * 0.90
energy_max = energy_target * 1.10

# Energy range
model += total_energy >= energy_min, "energy_min"
model += total_energy <= energy_max, "energy_max"

# Carbohydrate minimum
if constraints.get("carb_min_g") is not None:
    model += total_carb >= constraints["carb_min_g"], "carb_min"

# Carbohydrate maximum, if available
if constraints.get("carb_max_g") is not None:
    model += total_carb <= constraints["carb_max_g"], "carb_max"

# Fat minimum, if available
if constraints.get("fat_min_g") is not None:
    model += total_fat >= constraints["fat_min_g"], "fat_min"

# Fat maximum, if available
if constraints.get("fat_max_g") is not None:
    model += total_fat <= constraints["fat_max_g"], "fat_max"

# Sodium maximum
if constraints.get("sodium_max_mg") is not None:
    model += total_sodium <= constraints["sodium_max_mg"], "sodium_max"

# Fiber minimum
if constraints.get("fiber_min_g") is not None:
    model += total_fiber >= constraints["fiber_min_g"], "fiber_min"

In [14]:
# ============================================================
# 6. Objective function: minimize energy deviation
# ============================================================

energy_dev_pos = pulp.LpVariable("energy_dev_pos", lowBound=0)
energy_dev_neg = pulp.LpVariable("energy_dev_neg", lowBound=0)

model += total_energy - energy_target == energy_dev_pos - energy_dev_neg

# Objective:
# minimize energy deviation
# plus small penalty for sodium
model += (
    energy_dev_pos
    + energy_dev_neg
    + 0.01 * total_sodium
), "minimize_energy_deviation_and_sodium"

In [15]:
# ============================================================
# 7. Solve model
# ============================================================

solver = pulp.PULP_CBC_CMD(msg=True)
model.solve(solver)

print("Solver status:", pulp.LpStatus[model.status])
print("Objective value:", pulp.value(model.objective))

Solver status: Optimal
Objective value: 1.4409999999999998


In [16]:
selected_items = []

for i in milp_df.index:
    val = pulp.value(x[i])

    if val is not None and val > 0:
        row = milp_df.loc[i].copy()
        row["selected_portions"] = val * portion_step
        selected_items.append(row)

milp_menu = pd.DataFrame(selected_items)

print("Solver status:", pulp.LpStatus[model.status])
print("Number of selected foods:", len(milp_menu))

if not milp_menu.empty:
    display(
        milp_menu[[
            "food_name",
            "category_code",
            "urt",
            "gram_per_portion",
            "selected_portions",
            "energy_kcal_per_portion",
            "protein_g_per_portion",
            "fat_g_per_portion",
            "carb_g_per_portion",
            "sodium_mg_per_portion",
            "fiber_g_per_portion"
        ]]
    )
else:
    print("milp_menu is empty. The extraction variable may not be the same variable used in the solved model.")

Solver status: Optimal
Number of selected foods: 12


,food_name,category_code,urt,gram_per_portion,selected_portions,energy_kcal_per_portion,protein_g_per_portion,fat_g_per_portion,carb_g_per_portion,sodium_mg_per_portion,fiber_g_per_portion
0,"Bihun, mentah",MP,1/2 Gelas,50.0,2.0,174.00,2.350,0.050,41.050,2.50,0.600
1,"Makaroni, mentah",MP,1½ Gelas,50.0,2.0,176.50,4.350,0.200,39.350,2.50,2.450
6,"Kacang kedelai, segar",LN,2 Sendok Makan,25.0,0.5,71.50,7.550,3.900,7.525,7.00,0.725
12,"Tahu, mentah",LN,2 Potong Sedang,100.0,0.5,80.00,10.900,4.700,0.800,2.00,0.100
13,"Tempe kedelai, mentah",LN,2 potong sedang,50.0,2.0,100.50,10.400,4.400,6.750,4.50,0.700
35,"Labu siam, segar",S,1 gelas,100.0,1.0,30.00,0.600,0.100,6.700,2.70,6.200
38,"Rebung, segar",S,1 gelas,100.0,2.0,25.00,0.800,0.100,5.300,3.00,9.700
54,"Duku, segar",B,14 buah sedang,80.0,2.0,50.40,0.800,0.160,12.880,1.60,3.440
76,"Ayam, daging, segar",LH,1 potong sedang,40.0,1.0,119.20,7.280,10.000,0.000,43.60,0.000
77,"Sapi, daging, gemuk, segar",LH,1 potong sedang,35.0,2.0,95.55,6.125,7.700,0.000,32.55,0.000


In [17]:
milp_menu["selected_gram"] = (
    milp_menu["gram_per_portion"] * milp_menu["selected_portions"]
)

In [18]:
nutrient_cols = [
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

for col in nutrient_cols:
    total_col = col.replace("_per_portion", "_selected_total")
    milp_menu[total_col] = (
        milp_menu[col] * milp_menu["selected_portions"]
    )

display(
    milp_menu[[
        "food_name",
        "category_code",
        "urt",
        "gram_per_portion",
    ] ]
)

,food_name,category_code,urt,gram_per_portion
0,"Bihun, mentah",MP,1/2 Gelas,50.0
1,"Makaroni, mentah",MP,1½ Gelas,50.0
6,"Kacang kedelai, segar",LN,2 Sendok Makan,25.0
12,"Tahu, mentah",LN,2 Potong Sedang,100.0
13,"Tempe kedelai, mentah",LN,2 potong sedang,50.0
35,"Labu siam, segar",S,1 gelas,100.0
38,"Rebung, segar",S,1 gelas,100.0
54,"Duku, segar",B,14 buah sedang,80.0
76,"Ayam, daging, segar",LH,1 potong sedang,40.0
77,"Sapi, daging, gemuk, segar",LH,1 potong sedang,35.0


In [19]:
milp_menu.columns

Index(['food_code', 'food_name', 'source', 'water_g_100g', 'energy_kcal_100g',
       'protein_g_100g', 'fat_g_100g', 'carb_g_100g', 'fiber_g_100g',
       'ash_g_100g', 'calcium_mg_100g', 'phosphorus_mg_100g', 'iron_mg_100g',
       'sodium_mg_100g', 'potassium_mg_100g', 'copper_mg_100g', 'zinc_mg_100g',
       'retinol_mcg_100g', 'beta_carotene_mcg_100g', 'carotene_total_mcg_100g',
       'thiamin_mg_100g', 'riboflavin_mg_100g', 'niacin_mg_100g',
       'vitamin_c_mg_100g', 'bdd_percent', 'tkpi_main_group',
       'tkpi_group_code', 'category_code', 'category_name', 'category_source',
       'food_name_clean', 'food_name_match', 'food_name_match_original', 'urt',
       'urt_qty', 'urt_unit', 'gram_per_portion', 'food_name_std',
       'is_processed_code', 'exists_in_urt', 'exists_in_manual_whitelist',
       'remove_processed', 'energy_kcal_per_portion', 'protein_g_per_portion',
       'fat_g_per_portion', 'carb_g_per_portion', 'sodium_mg_per_portion',
       'potassium_mg_per_porti

In [20]:
total_summary = (
    milp_menu[
        [col.replace("_per_portion", "_selected_total") for col in nutrient_cols]
    ]
    .sum()
    .reset_index()
)

total_summary.columns = ["nutrient", "total"]

display(total_summary)

,nutrient,total
0,energy_kcal_selected_total,1600.0000
1,protein_g_selected_total,66.8550
2,fat_g_selected_total,54.4150
3,carb_g_selected_total,221.5225
4,sodium_mg_selected_total,144.1000
5,potassium_mg_selected_total,1538.9275
6,calcium_mg_selected_total,378.4000
7,fiber_g_selected_total,40.3925


In [21]:
total_energy_value = milp_menu["energy_kcal_selected_total"].sum()
total_protein_value = milp_menu["protein_g_selected_total"].sum()
total_fat_value = milp_menu["fat_g_selected_total"].sum()
total_carb_value = milp_menu["carb_g_selected_total"].sum()
total_sodium_value = milp_menu["sodium_mg_selected_total"].sum()
total_fiber_value = milp_menu["fiber_g_selected_total"].sum()

validation_rows = []

validation_rows.append({
    "constraint": "energy",
    "value": total_energy_value,
    "target_or_limit": f"{energy_min} - {energy_max}",
    "status": "pass" if energy_min <= total_energy_value <= energy_max else "fail"
})

if constraints.get("carb_min_g") is not None:
    validation_rows.append({
        "constraint": "carb_min",
        "value": total_carb_value,
        "target_or_limit": f">= {constraints['carb_min_g']}",
        "status": "pass" if total_carb_value >= constraints["carb_min_g"] else "fail"
    })

if constraints.get("carb_max_g") is not None:
    validation_rows.append({
        "constraint": "carb_max",
        "value": total_carb_value,
        "target_or_limit": f"<= {constraints['carb_max_g']}",
        "status": "pass" if total_carb_value <= constraints["carb_max_g"] else "fail"
    })

if constraints.get("sodium_max_mg") is not None:
    validation_rows.append({
        "constraint": "sodium_max",
        "value": total_sodium_value,
        "target_or_limit": f"<= {constraints['sodium_max_mg']}",
        "status": "pass" if total_sodium_value <= constraints["sodium_max_mg"] else "fail"
    })

if constraints.get("fiber_min_g") is not None:
    validation_rows.append({
        "constraint": "fiber_min",
        "value": total_fiber_value,
        "target_or_limit": f">= {constraints['fiber_min_g']}",
        "status": "pass" if total_fiber_value >= constraints["fiber_min_g"] else "fail"
    })

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

,constraint,value,target_or_limit,status
0,energy,1600.0000,1440.0 - 1760.0000000000002,pass
1,carb_min,221.5225,>= 130,pass
2,sodium_max,144.1000,<= 2000,pass
3,fiber_min,40.3925,>= 40.0,pass


In [22]:
portion_check = (
    milp_menu
    .groupby("category_code")["selected_portions"]
    .sum()
    .reset_index()
    .rename(columns={"selected_portions": "selected_portions_total"})
)

adjusted_plan_df = pd.DataFrame([
    {"category_code": k, "required_portions": v}
    for k, v in adjusted_plan.items()
])

portion_check = adjusted_plan_df.merge(
    portion_check,
    on="category_code",
    how="left"
).fillna(0)

portion_check["status"] = np.where(
    portion_check["required_portions"] == portion_check["selected_portions_total"],
    "pass",
    "fail"
)

display(portion_check)

,category_code,required_portions,selected_portions_total,status
0,MP,4,4.0,pass
1,LH,3,3.0,pass
2,LN,3,3.0,pass
3,S,3,3.0,pass
4,B,2,2.0,pass
5,SS,0,0.0,pass
6,M,3,3.0,pass
7,G,0,0.0,pass


In [23]:
# ============================================================
# 8. Evaluate menu against constraints
# ============================================================

def evaluate_menu(daily_totals, constraints):
    evaluation = []

    energy_total = daily_totals.get("energy_kcal_per_portion", np.nan)
    carb_total = daily_totals.get("carb_g_per_portion", np.nan)
    protein_total = daily_totals.get("protein_g_per_portion", np.nan)
    fat_total = daily_totals.get("fat_g_per_portion", np.nan)
    sodium_total = daily_totals.get("sodium_mg_per_portion", np.nan)
    potassium_total = daily_totals.get("potassium_mg_per_portion", np.nan)
    calcium_total = daily_totals.get("calcium_mg_per_portion", np.nan)
    fiber_total = daily_totals.get("fiber_g_per_portion", np.nan)

    # Energy: allow ±10% for rule-based prototype
    energy_target = constraints["energy_target"]
    energy_min = energy_target * 0.90
    energy_max = energy_target * 1.10

    evaluation.append({
        "constraint": "energy",
        "value": energy_total,
        "target_or_limit": f"{energy_min:.1f} - {energy_max:.1f}",
        "status": "pass" if energy_min <= energy_total <= energy_max else "not_pass"
    })

    # Carbohydrate minimum
    if constraints.get("carb_min_g") is not None:
        evaluation.append({
            "constraint": "carb_min",
            "value": carb_total,
            "target_or_limit": f">= {constraints['carb_min_g']:.1f}",
            "status": "pass" if carb_total >= constraints["carb_min_g"] else "not_pass"
        })

    # Carbohydrate maximum
    if constraints.get("carb_max_g") is not None:
        evaluation.append({
            "constraint": "carb_max",
            "value": carb_total,
            "target_or_limit": f"<= {constraints['carb_max_g']:.1f}",
            "status": "pass" if carb_total <= constraints["carb_max_g"] else "not_pass"
        })

    # Fat minimum
    if constraints.get("fat_min_g") is not None:
        evaluation.append({
            "constraint": "fat_min",
            "value": fat_total,
            "target_or_limit": f">= {constraints['fat_min_g']:.1f}",
            "status": "pass" if fat_total >= constraints["fat_min_g"] else "not_pass"
        })

    # Fat maximum
    if constraints.get("fat_max_g") is not None:
        evaluation.append({
            "constraint": "fat_max",
            "value": fat_total,
            "target_or_limit": f"<= {constraints['fat_max_g']:.1f}",
            "status": "pass" if fat_total <= constraints["fat_max_g"] else "not_pass"
        })

    # Sodium maximum
    if constraints.get("sodium_max_mg") is not None:
        evaluation.append({
            "constraint": "sodium_max",
            "value": sodium_total,
            "target_or_limit": f"<= {constraints['sodium_max_mg']:.1f}",
            "status": "pass" if sodium_total <= constraints["sodium_max_mg"] else "not_pass"
        })

    # Fiber minimum
    if constraints.get("fiber_min_g") is not None:
        evaluation.append({
            "constraint": "fiber_min",
            "value": fiber_total,
            "target_or_limit": f">= {constraints['fiber_min_g']:.1f}",
            "status": "pass" if fiber_total >= constraints["fiber_min_g"] else "not_pass"
        })

    # Potassium minimum
    if constraints.get("potassium_min_mg") is not None:
        evaluation.append({
            "constraint": "potassium_min",
            "value": potassium_total,
            "target_or_limit": f">= {constraints['potassium_min_mg']:.1f}",
            "status": "pass" if potassium_total >= constraints["potassium_min_mg"] else "not_pass"
        })

    # Calcium minimum
    if constraints.get("calcium_min_mg") is not None:
        evaluation.append({
            "constraint": "calcium_min",
            "value": calcium_total,
            "target_or_limit": f">= {constraints['calcium_min_mg']:.1f}",
            "status": "pass" if calcium_total >= constraints["calcium_min_mg"] else "not_pass"
        })

    return pd.DataFrame(evaluation)




In [24]:
# ============================================================
# 9. Calculate MILP nutrient totals
# ============================================================

nutrient_cols = [
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

existing_nutrient_cols = [
    col for col in nutrient_cols if col in milp_menu.columns
]

milp_totals = {}

for col in existing_nutrient_cols:
    milp_totals[col] = (
        milp_menu[col] * milp_menu["selected_portions"]
    ).sum()

milp_totals = pd.Series(milp_totals)

display(
    milp_totals
    .reset_index()
    .rename(columns={"index": "nutrient", 0: "total"})
)

,nutrient,total
0,energy_kcal_per_portion,1600.0000
1,protein_g_per_portion,66.8550
2,fat_g_per_portion,54.4150
3,carb_g_per_portion,221.5225
4,sodium_mg_per_portion,144.1000
5,potassium_mg_per_portion,1538.9275
6,calcium_mg_per_portion,378.4000
7,fiber_g_per_portion,40.3925


In [25]:
milp_evaluation = evaluate_menu(
    milp_totals,
    constraints
)

display(milp_evaluation)

,constraint,value,target_or_limit,status
0,energy,1600.0000,1440.0 - 1760.0,pass
1,carb_min,221.5225,>= 130.0,pass
2,sodium_max,144.1000,<= 2000.0,pass
3,fiber_min,40.3925,>= 40.0,pass


In [26]:
selected_category_count = (
    milp_menu
    .groupby("category_code")["selected_portions"]
    .sum()
    .reset_index(name="selected_portions")
)

display(selected_category_count)

print("Adjusted portion plan:")
print(adjusted_plan)

,category_code,selected_portions
0,B,2.0
1,LH,3.0
2,LN,3.0
3,M,3.0
4,MP,4.0
5,S,3.0


Adjusted portion plan:
{'MP': 4, 'LH': 3, 'LN': 3, 'S': 3, 'B': 2, 'SS': 0, 'M': 3, 'G': 0}


In [27]:
milp_menu.to_csv("milp_menu_dm_ht_obesity.csv", index=False)
milp_evaluation.to_csv("milp_menu_evaluation_dm_ht_obesity.csv", index=False)

milp_totals.reset_index().rename(
    columns={"index": "nutrient", 0: "total"}
).to_csv("milp_menu_nutrient_total_dm_ht_obesity.csv", index=False)

print("Saved:")
print("- milp_menu_dm_ht_obesity.csv")
print("- milp_menu_evaluation_dm_ht_obesity.csv")
print("- milp_menu_nutrient_total_dm_ht_obesity.csv")

Saved:
- milp_menu_dm_ht_obesity.csv
- milp_menu_evaluation_dm_ht_obesity.csv
- milp_menu_nutrient_total_dm_ht_obesity.csv


In [28]:
# ============================================================
# 3. Meal allocation rules
# ============================================================

meal_rules = {
    "MP": ["breakfast", "lunch", "dinner"],
    "LN": ["breakfast", "lunch", "dinner"],
    "LH": ["lunch", "dinner"],
    "S":  ["breakfast", "lunch", "dinner"],
    "B":  ["morning_snack", "afternoon_snack"],
    "M":  ["lunch", "dinner"],
    "SS": ["morning_snack", "afternoon_snack"],
    "G":  []
}

In [29]:
import pandas as pd
import numpy as np

# ==========================================
# 1. DAILY MENU RESULT FROM MILP
# ==========================================

# milp_menu
# assumed columns:
# - food_name
# - category_code
# - selected_portions
# - gram_per_portion
# - nutrient columns

menu = milp_menu.copy()

# ==========================================
# 2. MEAL DISTRIBUTION RULES
# ==========================================

meal_rules = {
    "MP": {  # makanan pokok
        "breakfast": 0.4,
        "lunch": 0.3,
        "dinner": 0.3,
    },
    "LH": {  # lauk hewani
        "lunch": 0.5,
        "dinner": 0.5,
    },
    "LN": {  # lauk nabati
        "breakfast": 0.3,
        "lunch": 0.3,
        "dinner": 0.4,
    },
    "S": {  # sayur
        "lunch": 0.5,
        "dinner": 0.5,
    },
    "B": {  # buah
        "morning_snack": 0.5,
        "afternoon_snack": 0.5,
    },
    "M": {  # minyak
        "lunch": 0.5,
        "dinner": 0.5,
    },
}

# ==========================================
# 3. ROUNDING STEP SIZE
# ==========================================

step_size = {
    "MP": 0.5,
    "LH": 0.5,
    "LN": 0.5,
    "S": 0.5,
    "B": 1.0,
    "M": 0.5,
}

# ==========================================
# 4. NUTRIENT COLUMNS
# ==========================================

nutrient_cols = [
    "energy_kcal_per_portion",
    "protein_g_per_portion",
    "fat_g_per_portion",
    "carb_g_per_portion",
    "sodium_mg_per_portion",
    "potassium_mg_per_portion",
    "calcium_mg_per_portion",
    "fiber_g_per_portion"
]

# ==========================================
# 5. DISTRIBUTE TO MEALS
# ==========================================

meal_rows = []

for _, row in menu.iterrows():

    category = row["category_code"]

    if category not in meal_rules:
        continue

    rules = meal_rules[category]
    step = step_size[category]

    for meal_name, ratio in rules.items():

        raw_portion = row["selected_portions"] * ratio

        # rounding to realistic step
        rounded_portion = round(raw_portion / step) * step

        if rounded_portion <= 0:
            continue

        meal_row = row.copy()

        meal_row["meal_time"] = meal_name
        meal_row["meal_portion"] = rounded_portion

        # grams
        meal_row["meal_gram"] = (
            rounded_portion * row["gram_per_portion"]
        )

        # URT display
        meal_row["meal_urt_qty"] = (
            rounded_portion * row["urt_qty"]
        )

        meal_row["meal_urt"] = (
            f"{meal_row['meal_urt_qty']} {row['urt_unit']}"
        )

        # nutrient calculation
        for col in nutrient_cols:

            meal_col = col.replace("_per_portion", "")

            meal_row[meal_col] = (
                rounded_portion * row[col]
            )

        meal_rows.append(meal_row)

meal_df = pd.DataFrame(meal_rows)

# ==========================================
# 6. RECALCULATE TOTALS
# ==========================================

summary_cols = [
    "energy_kcal",
    "protein_g",
    "fat_g",
    "carb_g",
    "sodium_mg",
    "fiber_g"
]

summary = (
    meal_df.groupby("meal_time")[summary_cols]
    .sum()
    .round(2)
)

print(summary)

# ==========================================
# 7. DAILY TOTAL VALIDATION
# ==========================================

daily_total = meal_df[summary_cols].sum().round(2)

print("\n=== DAILY TOTAL ===")
print(daily_total)

# ==========================================
# 8. SIMPLE REPAIR OPTIMIZATION
# ==========================================

patient_target = {
    "energy_kcal": 1600,
    "protein_g": 60,
    "fat_g": 44,
    "carb_g": 240,
    "fiber_g_min": 25,
    "sodium_mg_max": 2000,
}

print("\n=== VALIDATION ===")

print(
    "Energy:",
    daily_total["energy_kcal"],
    "/",
    patient_target["energy_kcal"]
)

print(
    "Protein:",
    daily_total["protein_g"],
    "/",
    patient_target["protein_g"]
)

print(
    "Fat:",
    daily_total["fat_g"],
    "/",
    patient_target["fat_g"]
)

print(
    "Carb:",
    daily_total["carb_g"],
    "/",
    patient_target["carb_g"]
)

print(
    "Fiber:",
    daily_total["fiber_g"],
    "/",
    patient_target["fiber_g_min"]
)

print(
    "Sodium:",
    daily_total["sodium_mg"],
    "/",
    patient_target["sodium_mg_max"]
)

# ==========================================
# 9. OPTIONAL:
# GREEDY REPAIR IF ENERGY TOO LOW
# ==========================================

energy_gap = (
    patient_target["energy_kcal"]
    - daily_total["energy_kcal"]
)

if energy_gap > 100:

    print("\nEnergy too low -> adding food")

    candidate = (
        menu.sort_values(
            "energy_kcal_per_portion",
            ascending=False
        )
        .iloc[0]
    )

    print(
        "Suggested additional food:",
        candidate["food_name"]
    )

# ==========================================
# 10. FINAL OUTPUT
# ==========================================

display(
    meal_df[[
        "meal_time",
        "food_name",
        "meal_portion",
        "meal_urt",
        "meal_gram",
        "energy_kcal",
        "protein_g",
        "fat_g",
        "carb_g",
        "fiber_g"
    ]]
    .sort_values(["meal_time"])
)

                 energy_kcal  protein_g  fat_g  carb_g  sodium_mg  fiber_g
meal_time                                                                 
afternoon_snack        50.40       0.80   0.16   12.88       1.60     3.44
breakfast             400.75      11.90   2.45   83.78       7.25     3.40
dinner                536.48      24.66  24.77   55.60      65.70    15.02
lunch                 486.22      19.46  22.57   52.22      63.45    14.68
morning_snack          50.40       0.80   0.16   12.88       1.60     3.44

=== DAILY TOTAL ===
energy_kcal    1524.25
protein_g        57.63
fat_g            50.12
carb_g          217.36
sodium_mg       139.60
fiber_g          39.98
dtype: float64

=== VALIDATION ===
Energy: 1524.25 / 1600
Protein: 57.63 / 60
Fat: 50.12 / 44
Carb: 217.36 / 240
Fiber: 39.98 / 25
Sodium: 139.6 / 2000


,meal_time,food_name,meal_portion,meal_urt,meal_gram,energy_kcal,protein_g,fat_g,carb_g,fiber_g
54,afternoon_snack,"Duku, segar",1.0,14.0 buah sedang,80.0,50.400,0.800,0.1600,12.880,3.440
0,breakfast,"Bihun, mentah",1.0,0.5 Gelas,50.0,174.000,2.350,0.0500,41.050,0.600
1,breakfast,"Makaroni, mentah",1.0,1.5 Gelas,50.0,176.500,4.350,0.2000,39.350,2.450
13,breakfast,"Tempe kedelai, mentah",0.5,1.0 potong sedang,25.0,50.250,5.200,2.2000,3.375,0.350
89,dinner,Minyak kedelai,0.5,0.5 sendok teh,2.5,22.075,0.000,2.4975,0.000,0.000
77,dinner,"Sapi, daging, gemuk, segar",1.0,1.0 potong sedang,35.0,95.550,6.125,7.7000,0.000,0.000
76,dinner,"Ayam, daging, segar",0.5,0.5 potong sedang,20.0,59.600,3.640,5.0000,0.000,0.000
38,dinner,"Rebung, segar",1.0,1.0 gelas,100.0,25.000,0.800,0.1000,5.300,9.700
35,dinner,"Labu siam, segar",0.5,0.5 gelas,50.0,15.000,0.300,0.0500,3.350,3.100
90,dinner,Minyak kelapa,1.0,1.0 sendok teh,5.0,43.500,0.050,4.9000,0.000,0.000
